# Entraînement du Modèle de Classification d'Anacardes (NIANKA IA)
Ce notebook permet d'entraîner un modèle de Deep Learning (MobileNetV3) avec Transfer Learning et Data Augmentation pour classifier les noix d'anacarde en 4 catégories : `grade_A`, `grade_B`, `grade_C`, et `rejete`.

### Étape 1 : Connexion à Google Drive et Décompression du Dataset

In [ ]:
from google.colab import drive
import zipfile
import os

# Monter Google Drive
drive.mount('/content/drive')

# Chemin vers le fichier zip sur votre Drive
zip_path = '/content/drive/MyDrive/databases.zip'
extract_dir = '/content/dataset'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print('✅ Décompression réussie dans :', extract_dir)
else:
    print('⚠️ Fichier databases.zip non trouvé dans MyDrive. Assurez-vous d'avoir téléversé databases.zip dans votre Google Drive, ou modifiez le chemin si vous l'avez mis dans un sous-dossier.')


### Étape 2 : Chargement des images et Data Augmentation dynamique

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 4
EPOCHS = 30

# Data Augmentation en temps réel durant l'entraînement
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1),
])

# Chargement des images
train_ds = tf.keras.utils.image_dataset_from_directory(
    extract_dir,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    extract_dir,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print('Classes identifiées :', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y)).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


### Étape 3 : Construction du Modèle (Transfer Learning MobileNetV3)

In [ ]:
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


### Étape 4 : Lancement de l'Entraînement

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


### Étape 5 : Sauvegarde et Téléchargement du Modèle Final

In [ ]:
from google.colab import files

model_path = '/content/model_anacarde.keras'
model.save(model_path)
print('✅ Modèle sauvegardé sous :', model_path)

files.download(model_path)
